In [1]:
from atomate2.forcefields.flows.phonons import PhononMaker
from pymatgen.core.structure import Structure


/home/a.burov/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/a.burov/.local/lib/python3.12/site-packages/pybtex/plugin/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
from pymatgen.phonon.bandstructure import PhononBandStructureSymmLine
from pymatgen.phonon.dos import PhononDos
from pymatgen.phonon.plotter import PhononBSPlotter, PhononDosPlotter
from jobflow import SETTINGS

In [17]:
import os
import numpy as np


In [4]:
os.environ["CHGNET_DISABLE_CUDA"] = "1"  # Force CPU


In [ ]:
def calc_phonopy_atomate2(equilibrium_atoms, model, mul_matrix=[[2,0,0], [0,2,0], [0,0,2]], 
                         env_used="msdb", scale=[0.97, 1.10], nsteps=300):
    """
    Gibbs free energy via QHA using atomate2 + CHGNet, matching your exact workflow.
    """
    from atomate2.forcefields.jobs import ForceFieldRelaxMaker, ForceFieldStaticMaker
    from atomate2.phonons.flows.phonon import PhononFlow
    from atomate2.phonons.flows.qha import QHAFlow
    from jobflow import run_locally
    from pymatgen.core import Structure
    
    # Convert ASE atoms to pymatgen Structure
    equilibrium_structure = AseAtomsAdaptor.get_structure(equilibrium_atoms)
    
    print("Starting QHA calculation with atomate2 + CHGNet...")
    
    # Define CHGNet makers (matches your model)
    relax_maker = ForceFieldRelaxMaker(
        force_field_name="CHGNet",
        relax_kwargs={"fmax": 0.001, "steps": nsteps}  # Your convergence criteria
    )
    static_maker = ForceFieldStaticMaker(force_field_name="CHGNet")
    
    # Full QHA workflow
    qha_flow = QHAFlow(
        relax_maker=relax_maker,
        phonon_maker=PhononFlow(
            relax_maker=relax_maker,
            static_maker=static_maker,
            min_length=10.0,  # Supercell size (equiv to your mul_matrix)
            displacement=0.02  # Your displacement distance
        ),
        volumes_range=(*scale, 13),  # scale[0], scale[1], 13 points (adjust as needed)
        t_max=1000,  # K
        t_step=10
    ).make(equilibrium_structure)
    
    # Execute workflow locally (SLURM via srun in production)
    print("Running atomate2 QHA workflow...")
    results = run_locally(qha_flow, create_folders=True)
    
    # Extract Gibbs free energy
    final_result = results[-1].output
    gibbs_data = final_result.minimized_gibbs_free_energy
    
    print("✅ QHA complete!")
    print(f"Gibbs free energy range: {gibbs_data['energy'].min():.6f} to {gibbs_data['energy'].max():.6f} eV/atom")
    
    # Save traditional outputs for your phonopy pipeline
    np.savetxt("gibbs_free_energy.dat", 
               np.column_stack([gibbs_data["T"], gibbs_data["energy"]]),
               header="# T(K)  G(T)(eV/atom)")
    
    return gibbs_data["T"], gibbs_data["energy"]

# Usage (exactly matches your signature):
# model = CHGNet.load()  # Your pretrained model
# atoms = read("POSCAR")
# T, G = calc_phonopy_atomate2(atoms, model, env_used="chgnet-env")


In [5]:
from jobflow import run_locally
from pymatgen.core import Structure
from atomate2.forcefields.jobs import ForceFieldRelaxMaker, ForceFieldStaticMaker
from atomate2.forcefields.flows.phonons import PhononMaker  # ✅ Correct import
from atomate2.vasp.flows.qha import QhaMaker


In [6]:
# CHGNet makers
chgnet_relax = ForceFieldRelaxMaker(
    force_field_name="CHGNet",
    relax_kwargs={"fmax": 0.01},
    # Add device override if supported
    calculator_kwargs={"device": "cpu"}
)


In [10]:
chgnet_static = ForceFieldRelaxMaker(
    force_field_name="CHGNet",
    relax_kwargs={"fmax": 0.01},
    # Add device override if supported
    calculator_kwargs={"device": "cpu"}
)


In [11]:
si_structure = Structure.from_file("/home/a.burov/icys_2025/niohf/optimized/chgnet/layered_p-1.vasp")


In [15]:
si_structure

Structure Summary
Lattice
    abc : 5.36450692291294 6.365873243987889 5.1016625652863805
 angles : 89.94319927947987 112.6214179474543 89.98119770664454
 volume : 160.8170806481406
      A : np.float64(4.949560034009389) np.float64(0.0008759240455304) np.float64(-2.0687651457995195)
      B : np.float64(0.0024557237537134) np.float64(6.365871989175168) np.float64(0.0031536291369434)
      C : np.float64(0.0055217610547012) np.float64(0.002528109450907) np.float64(5.101658950661233)
    pbc : True True True
PeriodicSite: H (3.816, 0.7922, 1.337) [0.7702, 0.1241, 0.5743]
PeriodicSite: H (3.849, 3.985, 1.346) [0.7767, 0.6257, 0.5784]
PeriodicSite: H (1.142, 5.577, 1.699) [0.2298, 0.8759, 0.4257]
PeriodicSite: H (1.109, 2.384, 1.69) [0.2233, 0.3743, 0.4216]
PeriodicSite: F (3.27, 0.7909, -0.2991) [0.6603, 0.1241, 0.2091]
PeriodicSite: F (1.688, 5.578, 3.335) [0.3397, 0.8759, 0.7909]
PeriodicSite: F (3.274, 3.981, -0.288) [0.6609, 0.6252, 0.2111]
PeriodicSite: F (1.684, 2.388, 3.324) [0.33

In [18]:
SUPERCELL_MATRIX = np.array([[2, 0, 0], 
                            [0, 2, 0], 
                            [0, 0, 2]])  # 8x atoms - bulletproof



In [13]:
# CHGNet phonon maker (matches your original example structure)
chgnet_phonon_maker = PhononMaker(
    displacement=0.01,           # Your original distance
    min_length=9.5,              # Safe (was 20.0 → too big!)
    max_length=None,
    sym_reduce=True,             # Symmetry reduction (fewer calcs)
    symprec=0.001,
    prefer_90_degrees=True,      # Nicer supercells
    allow_orthorhombic=False,    # Stick to your mul_matrix style
    bulk_relax_maker=None,       # Skip (already relaxed)
    static_energy_maker=chgnet_static,
    phonon_displacement_maker=chgnet_static,
    generate_frequencies_eigenvectors_kwargs={
        "tmin": 0,
        "tmax": 1000, 
        "tstep": 10
    },
    use_symmetrized_structure=None  # Keep your input as-is
)


# CRITICAL: Disable auto-supercell + use your matrix
phonon_maker = PhononMaker(
    min_length=None,              
    max_length=None,                
    supercell_matrix=SUPERCELL_MATRIX,  # 
    displacement=0.01,
    sym_reduce=True,
    # symprec=0.001,
    static_energy_maker=chgnet_static,
    phonon_displacement_maker=chgnet_static,
    generate_frequencies_eigenvectors_kwargs={
        "tmin": 0, "tmax": 1000, "tstep": 10
    }
)

In [14]:
flow = QhaMaker(
    initial_relax_maker=chgnet_relax,
    eos_relax_maker=chgnet_relax,
    phonon_maker=chgnet_phonon_maker,
    linear_strain=(-0.03, 0.10),    # ±15% volume scan
    number_of_frames=6,             # 6 points
    pressure=None,
    t_max=1000,
    eos_type="vinet",
).make(structure=si_structure)



In [15]:
results = run_locally(flow, create_folders=True)


2026-02-06 20:31:21,088 INFO Started executing jobs locally
2026-02-06 20:31:21,106 INFO Starting job - EOS equilibrium relaxation (a77bb61e-02d7-4f1c-a8d0-e360f759e990)
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cpu
2026-02-06 20:32:31,418 INFO Finished job - EOS equilibrium relaxation (a77bb61e-02d7-4f1c-a8d0-e360f759e990)
2026-02-06 20:32:31,423 INFO Starting job - apply_strain_to_structure (c5c658de-aa1b-48f0-8c20-53cea3953310)
2026-02-06 20:32:32,468 INFO Finished job - apply_strain_to_structure (c5c658de-aa1b-48f0-8c20-53cea3953310)
2026-02-06 20:32:32,469 INFO Starting job - Force field relax deformation 0 (a85c8df7-cc87-4991-b6b8-4d6ec8769db5)
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on cpu
2026-02-06 20:33:25,714 INFO Finished job - Force field relax deformation 0 (a85c8df7-cc87-4991-b6b8-4d6ec8769db5)
2026-02-06 20:33:25,719 INFO Starting job - Force field relax deformation 1 (41c97dd6-4804-4b7a-81a1-017779086ccf)
CHGNet v0.3

In [ ]:
from jobflow import run_locally
import numpy as np
import matplotlib.pyplot as plt
from pymatgen.phonon.bandstructure import PhononBandStructureSymmLine
from pymatgen.phonon.dos import PhononDos
from pymatgen.phonon.plotter import PhononBSPlotter, PhononDosPlotter
import json


In [ ]:
# After running your QHA flow:
results = run_locally(flow, create_folders=True)
final_result = results[-1].output

## 1. Save Gibbs Free Energy (primary output)
gibbs = final_result.minimized_gibbs_free_energy  # dict: {'T': [...], 'energy': [...]}
np.savetxt("gibbs_free_energy.dat", 
           np.column_stack([gibbs['T'], gibbs['energy']]),
           header="# T(K)  G(T)(eV/atom)", fmt="%.6f")

# JSON for downstream workflows
gibbs_json = {'T_K': gibbs['T'].tolist(), 'G_eV_per_atom': gibbs['energy'].tolist()}
with open("gibbs_free_energy.json", 'w') as f:
    json.dump(gibbs_json, f, indent=2)

## 2. Save Full QHA Data
np.savetxt("gibbs_vs_volume.dat", 
           np.column_stack([final_result.gibbs_free_energy['T'], 
                           final_result.gibbs_free_energy['energy']]),
           header="# T(K)  G(T,V)(eV)", fmt="%.6f")

## 3. Generate & Save Plots
# Gibbs free energy vs T
plt.figure(figsize=(8, 6))
plt.plot(gibbs['T'], gibbs['energy'], 'b-', linewidth=2, label='CHGNet QHA')
plt.xlabel('Temperature (K)')
plt.ylabel('Gibbs Free Energy (eV/atom)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig("gibbs_free_energy.png", dpi=300, bbox_inches='tight')
plt.show()

# Phonon band structure (equilibrium volume)
ph_bs = PhononBandStructureSymmLine.from_dict(final_result.phonon_bandstructure)
plotter = PhononBSPlotter(ph_bs)
plotter.get_plot().savefig("phonon_bandstructure.png", dpi=300, bbox_inches='tight')
plotter.show()

# Phonon DOS
ph_dos = PhononDos.from_dict(final_result.phonon_dos)
dos_plotter = PhononDosPlotter()
dos_plotter.add_dos("CHGNet", ph_dos)
dos_plotter.get_plot().savefig("phonon_dos.png", dpi=300, bbox_inches='tight')
dos_plotter.show()

# Thermal expansion coefficient α(T)
thermal_exp = final_result.thermal_expansion
plt.figure(figsize=(8, 6))
plt.plot(thermal_exp['T'], thermal_exp['alpha'], 'r-', linewidth=2)
plt.xlabel('Temperature (K)')
plt.ylabel('Thermal Expansion Coefficient (1/K)')
plt.grid(True, alpha=0.3)
plt.savefig("thermal_expansion.png", dpi=300, bbox_inches='tight')
plt.show()

print("Saved:")
print("gibbs_free_energy.dat     (T, G)")
print("gibbs_free_energy.json    (JSON)")
print("gibbs_free_energy.png     (G vs T)")
print("phonon_bandstructure.png  (bands)")
print("phonon_dos.png           (DOS)")
print("thermal_expansion.png    (α(T))")


## Internal phonopy methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.optimize import FIRE
from ase import Atoms
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
import yaml
import subprocess

from ase.calculators.eam import EAM

import os
import shutil
import glob
from pathlib import Path

from ase.build import bulk
# from ase.constraints import ExpCellFilter

import pandas as pd
import yaml



In [ ]:
from pymatgen.io.ase import AseAtomsAdaptor
from chgnet.model import StructOptimizer
from chgnet.model import CHGNet



In [ ]:
from phonopy import Phonopy
from phonopy.interface.calculator import get_displacements_and_forces
from phonopy.structure.atoms import PhonopyAtoms


In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
np.set_printoptions(precision=4)


In [ ]:
# %matplotlib inline
plt.rcParams['figure.dpi'] = 450


In [ ]:
chgnet_cal = StructOptimizer(use_device='cpu', optimizer_class="FIRE", )



In [ ]:
model = CHGNet.load(model_name="0.3.0", use_device="cpu")             # For forces


In [ ]:
fontsize = 20

In [ ]:
env_used = "chgnet-env"

In [ ]:
potential = "chgnet"

In [ ]:
def calc_phonopy(equilibrium_atoms, model, mul_matrix=[[2,0,0], [0,2,0], [0,0,2]], 
                env_used="msdb", scale=[0.97, 1.10], nsteps=300):
    
    # Volume scaling
    volume_points = 6
    scale_init = scale[0]
    scale_end = scale[1]
    volume_scales = np.linspace(scale_init, scale_end, volume_points)
    volumes = []
    energies = []
    
    print("Starting QHA calculation with CHGNet...")
    
    # Compute phonons at volume points
    for i, scale_factor in enumerate(volume_scales):
        print(f"\n{'='*60}")
        print(f"Volume point {i+1}/{volume_points}: scale = {scale_factor:.4f}")
        print(f"{'='*60}")
        
        # Scale structure
        scaled_atoms = equilibrium_atoms.copy()
        scaled_atoms.set_cell(equilibrium_atoms.get_cell() * scale_factor**(1/3), scale_atoms=True)
        
        # Relax with CHGNet
        structure = AseAtomsAdaptor.get_structure(scaled_atoms)
        pred = model.predict_structure(structure)
        en = pred['e']
        
        for step in range(nsteps):
            forces = pred['f']
            fmax = np.max(np.linalg.norm(forces, axis=1))
            if fmax < 0.001:
                print(f"Converged after {step+1} steps (fmax={fmax:.4f})")
                break
            
            displacement = -forces * 0.05
            scaled_atoms.set_positions(scaled_atoms.get_positions() + displacement)
            structure = AseAtomsAdaptor.get_structure(scaled_atoms)
            pred = model.predict_structure(structure)
        
        vol = scaled_atoms.get_volume()
        energies.append(en)
        volumes.append(vol)
        print(f"Volume: {vol:.4f} Å³, Energy: {en:.6f} eV")
        
        # Phonopy setup
        unitcell = PhonopyAtoms(
            symbols=scaled_atoms.get_chemical_symbols(),
            cell=scaled_atoms.get_cell(),
            scaled_positions=scaled_atoms.get_scaled_positions()
        )
        phonon = Phonopy(unitcell, supercell_matrix=mul_matrix)
        phonon.generate_displacements(distance=0.02)
        
        # Forces for displacements
        supercells = phonon.supercells_with_displacements
        sets_of_forces = []
        print(f"Computing {len(supercells)} displacements...")
        
        for j, scell in enumerate(supercells):
            atoms_disp = Atoms(symbols=scell.symbols, positions=scell.positions, 
                             cell=scell.cell, pbc=True)
            struct_disp = AseAtomsAdaptor.get_structure(atoms_disp)
            pred = model.predict_structure(struct_disp)
            forces = pred['f']
            sets_of_forces.append(forces)
        
        phonon.forces = sets_of_forces
        phonon.produce_force_constants()
        phonon.run_mesh([50, 50, 50])
        phonon.run_thermal_properties(t_step=10, t_max=1500, t_min=0)
        phonon.write_yaml_thermal_properties(f"thermal_properties.yaml-{i:02d}")
    
    # Save E-V data
    np.savetxt("e-v.dat", np.column_stack([volumes, energies]), fmt="%.8f", 
               header="# volume(Å³)  energy(eV)")
    
    # NO CLEANING NEEDED - phonopy-qha handles units automatically!
    thermal_files = sorted(glob.glob("thermal_properties.yaml-*"))
    print(f"Found {len(thermal_files)} thermal files - running QHA directly")
    
    # Run QHA
    phonopy_qha_path = f'/home/a.burov/micromamba/envs/{env_used}/bin/phonopy-qha'
    cmd = f"{phonopy_qha_path} -p -s --tmax=1000 e-v.dat {' '.join(thermal_files)}"
    print(f"\nRunning QHA: {cmd}")
    
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ QHA complete!")
        print("Output files: helmholtz-volume.dat, thermal_properties-P.dat")
    else:
        print("❌ QHA failed:")
        print(result.stderr)
        print(result.stdout)
    
    return volumes, energies


In [ ]:
def visualize_data(potential, structure):
    # FIRST PLOT
    # Load the fitted data (starts after "# Fitted data")
    with open("helmholtz-volume_fitted.dat") as f:
        lines = f.readlines()
    
    # Find where fitted data starts
    fitted_start = None
    minima_start = None
    for i, line in enumerate(lines):
        if "# Fitted data" in line:
            fitted_start = i + 1
        if "# Minimas" in line:
            minima_start = i + 1
            break
    
    # Read fitted data
    fitted_data = []
    for line in lines[fitted_start:minima_start-1]:
        if line.strip() and not line.startswith('#'):
            fitted_data.append([float(x) for x in line.split()])
    
    fitted_data = np.array(fitted_data)
    volumes_fitted = fitted_data[:, 0]
    helmholtz_fitted = fitted_data[:, 1:]  # All temperature columns
    
    # Read minima (equilibrium points)
    minima_data = []
    for line in lines[minima_start:]:
        if line.strip() and not line.startswith('#'):
            minima_data.append([float(x) for x in line.split()])
    
    minima_data = np.array(minima_data)
    vol_eq = minima_data[:, 0]
    energy_eq = minima_data[:, 1]
    
    # Number of temperatures
    n_temps = helmholtz_fitted.shape[1]
    print(f"Number of temperatures: {n_temps}")
    print(f"Number of volume points: {len(volumes_fitted)}")
    print(f"Equilibrium points: {len(vol_eq)}")
    
    # Create plot like your reference image
    fig, ax = plt.subplots(figsize=(8, 10))
    
    N = 5
    # Plot all temperature curves (blue)
    for i in range(n_temps):
        ax.plot(volumes_fitted[::N], helmholtz_fitted[::N, i], 'o-', color='blue', linewidth=1.5, markersize=4, alpha=0.8)
    
    # Plot equilibrium line (red)
    ax.plot(vol_eq, energy_eq, '*-', color='red', linewidth=1.5, markersize=10, zorder=10)
    
    # Formatting
    ax.set_xlabel('Volume (Å³)', fontsize=fontsize)
    ax.set_ylabel('Free energy (eV)', fontsize=fontsize)
    ax.tick_params(labelsize=fontsize-2)
    ax.grid(False)
    
    plt.tight_layout()
    plt.show()
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_free_{structure}.png", dpi=450)
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_free_{structure}.pdf", dpi=450)
    plt.close()
    
    print(f"Plot saved to: /home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_{structure}")
    
    # Print temperature info
    print(f"\nEquilibrium volumes:")
    print(f"  At T=0 K:   {vol_eq[0]:.4f} Å³")
    print(f"  At T={n_temps-1}00 K: {vol_eq[-1]:.4f} Å³")
    print(f"  Expansion: {vol_eq[-1] - vol_eq[0]:.4f} Å³")

    # SECOND PLOT
    # Load thermal_expansion.dat
    data = np.loadtxt("thermal_expansion.dat")
    temperatures = data[:, 0]
    alpha = data[:, 1]  # Thermal expansion coefficient (1/K)
    
    # Set font size
    
    # Create plot
    fig, ax = plt.subplots(figsize=(8, 6))
    
    ax.plot(temperatures, alpha, '-', color='red', linewidth=2.5)
    
    ax.set_xlabel('Temperature (K)', fontsize=fontsize)
    ax.set_ylabel(r'Thermal expansion (K$^{-1}$)', fontsize=fontsize)
    ax.tick_params(labelsize=fontsize-2)
    ax.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    ax.yaxis.offsetText.set_fontsize(fontsize-2)  # 
    
    plt.tight_layout()
    plt.show()
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_thermal_{structure}.pdf", dpi=450)
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_thermal_{structure}.png", dpi=450)
    plt.close()
    
    print(f"✓ Thermal expansion coefficient plot saved")
    print(f"\nα at 0 K: {alpha[0]:.2e} ")
    print(f"α at 300 K: {alpha[30]:.2e}")
    print(f"α at 1000 K: {alpha[100]:.2e}")
    
    # THIRD PLOT
    # Load data
    data = np.loadtxt("gibbs-temperature.dat")
    temperatures = data[:, 0]
    gibbs = data[:, 1]  # kJ/mol
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    ax.plot(temperatures, gibbs, '-', color='red', linewidth=2.5)
    
    ax.set_xlabel('Temperature (K)', fontsize=fontsize)
    ax.set_ylabel('Gibbs free energy (eV)', fontsize=fontsize)
    ax.tick_params(labelsize=fontsize-2)
    # ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_gibbs_{structure}.png", dpi=450)
    plt.savefig(f"/home/a.burov/icys_2025/niohf/data/figures/phonons/{potential}_gibbs_{structure}.png", dpi=450)
    plt.close()
    
    print(f"✓ Gibbs free energy plot saved")
    print(f"\nG at 0 K:    {gibbs[0]:.2f} kJ/mol")
    print(f"G at 300 K:  {gibbs[30]:.2f} kJ/mol")
    print(f"G at 1000 K: {gibbs[100]:.2f} kJ/mol")
    

    
    


In [ ]:
def save_files(path_save):
    # create destination directory
    Path(f"{path_save}").mkdir(parents=True, exist_ok=True)

    # Define patterns to match
    patterns = [
        "*.pdf",
        "*.dat", 
        "*.png",
        "*.log",
        "*.traj",
        "thermal_properties.yaml*"
    ]
    
    # Move all matching files
    moved_count = 0
    for pattern in patterns: 
        if (pattern != f"{potential}.log"):
            files = glob.glob(pattern)
            for file in files:
                dest_path = os.path.join(path_save, os.path.basename(file))
                
                # Remove destination file if it exists
                if os.path.exists(dest_path):
                    os.remove(dest_path)
                
                shutil.move(file, dest_path)  # Changed: use dest_path instead of path_save
                # print(f"Moved: {file} -> {dest_path}")
                moved_count += 1
    
    print(f"\n✓ Total files moved: {moved_count}")


In [ ]:
# Load equilibrium structure
# equilibrium_atoms = read("/home/a.burov/icys_2025/niohf/optimized/dft/delta.cif")


In [ ]:
# equilibrium_atoms = bulk('Al', 'fcc', a=4.05)



In [ ]:
path_base = "/home/a.burov/icys_2025/niohf/optimized/chgnet/"

In [ ]:
structures_phases = [
    "layered_p1", 
    "layered_p-1",
    "layered_p21",
    "layered_p2c",
    "layered_p21c",
    
    "alpha", 
    "beta", 
    "gamma", 
    "delta",      
]


In [ ]:
for st in structures_phases:
    if "layered" in st:
        mul_matrix=[[2,0,0], [0,2,0], [0,0,2]]
        scale = [0.97, 1.10]
    else:
        mul_matrix=[[4,0,0], [0,3,0], [0,0,1]]
        scale = [0.97, 1.08]
         
    
    path_file = f"{path_base}/{st}.cif"  
    # read relaxed structures
    equilibrium_atoms = read(path_file)
    # equilibrium_atoms = bulk('Al', 'fcc', a=4.05)

    # path to save phonopy data
    path_data_phonopy = f"/home/a.burov/icys_2025/niohf/umlip_phonons/{potential}/{st}"

    # perform phonopy calculations
    calc_phonopy(equilibrium_atoms, model=model, env_used=env_used, mul_matrix=mul_matrix, scale=scale)

    # visualize data and save plots
    visualize_data(potential=potential, structure=st)

    # save phonopy files
    save_files(path_data_phonopy)

    print("Finished without errors!")


    